In [0]:
from pyspark.sql import functions as f
from pyspark.sql.types import StringType

In [0]:
%run ../functions/functions

In [0]:

database_name = "dimensao"
table_name = "dm_uf"
target_path = f"{database_name}.{table_name}"
pk = "SK_SIMPLES"

In [0]:


container_destino = "gold"
 
container_origem = "silver"
caminho_origem = f"abfss://{container_origem}@{STORAGE}.dfs.core.windows.net/balancacomercial"

In [0]:
df_regiao = spark.read.format("delta").load(f"{caminho_origem}/UF_MUN")
df_regiao_2 = spark.read.format("delta").load(f"{caminho_origem}/UF")
df_importacao = spark.read.format("delta").load(f"{caminho_origem}/IMPORTACAO_MUN_CONSOLIDADA")
df_paises = spark.read.format("delta").load(f"{caminho_origem}/PAIS")
df_blocos = spark.read.format("delta").load(f"{caminho_origem}/PAIS_BLOCO")

df_final = df_regiao.select(
    "CO_MUN_GEO",
    "NO_MUN",
    "SG_UF"
)

df_join = df_final.join(df_regiao_2, "SG_UF", "left")

df_ponte = df_join.join(
    df_importacao,
    df_join.SG_UF == df_importacao.SG_UF_MUN, 
    "inner" 
)


df_com_pais = df_ponte.join(
    df_paises,
    df_ponte.CO_PAIS == df_paises.CO_PAIS,
    "left"
).drop(df_paises.CO_PAIS)


df_completo = df_com_pais.join(
    df_blocos,
    df_com_pais.CO_PAIS == df_blocos.CO_PAIS,
    "left"
).drop(df_blocos.CO_PAIS)




df_resultado_final = df_completo.select(
    # f.col("SK_UF"),
    # f.concat(df_join.SG_UF, df_join.NO_MUN).alias("sk_empresas_regioes"),
    # f.concat(df_join.SG_UF,df_join.CO_MUN_GEO).alias("sk_estabelecimentos_regioes"),
    df_join.CO_MUN_GEO,     
    df_join.NO_MUN,         
    df_join.SG_UF,          
    f.col("NO_REGIAO"),   
    f.col("CO_PAIS"),       
    f.col("NO_PAIS"),       
    f.col("NO_BLOCO")     
)

df_resultado_final.display()

In [0]:

if not spark.catalog.tableExists(target_path):
    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_path)
else:
    target_table = DeltaTable.forName(spark, target_path)
    target_table.alias("target").merge(
        df_final.alias("source"),
        f"target.{pk} = source.{pk}"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()